# Oracle 테이블 현황 조사 — survey_v1 (Jupyter)

셀을 위에서 아래로 순서대로 실행합니다.  
셀 7이 가장 오래 걸립니다. 완료 후 셀 8로 체크포인트를 저장하면  
다음 실행 시 셀 9로 로드해 Excel 출력(셀 10~11)만 재실행할 수 있습니다.

| 셀 | 내용 | 비고 |
|---|---|---|
| 1 | 임포트 | |
| 2 | **설정 ★ 수정 필요** | DB접속정보, 파일경로 |
| 3 | 유틸리티 함수 | |
| 4 | DB 쿼리 함수 | |
| 5 | Oracle 접속 | |
| 6 | 사용여부 맵 로드 | |
| 7 | **테이블 데이터 수집** | ⏳ 오래 걸림 |
| 8 | 체크포인트 저장 | |
| 9 | 체크포인트 로드 (선택) | 재실행 시 셀 7 건너뛰기 |
| 10 | 메인 데이터 → Excel | |
| 11 | **샘플 시트 → Excel** | ⏳ 오래 걸림 |

In [ ]:
# 셀 1: 임포트
import os, re, sys, shutil, pickle
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import oracledb
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils import column_index_from_string
from dateutil import parser as dateutil_parser

print('imports OK')

In [ ]:
# 셀 2: 설정 ★ 아래 값을 실제 환경에 맞게 수정하세요
ORACLE_CLIENT_LIB = r'C:\oracle\instantclient_21_9'  # ★

DB_HOST     = 'PROD_HOST'      # ★
DB_PORT     = 1521
DB_SERVICE  = 'PROD_SERVICE'   # ★
DB_USER     = 'PROD_USER'      # ★
DB_PASSWORD = 'PROD_PASSWORD'  # ★

TEMPLATE_EXCEL  = r'C:\path\to\template.xlsx'  # ★
START_CELL      = 'B4'
SYSTEM_NAME     = 'TEST'  # ★
SAMPLE_ROWS     = 10

CHECKPOINT_FILE = 'survey_v1_checkpoint.pkl'

In [ ]:
# 셀 3: 유틸리티 함수
ILLEGAL_CHARS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
CREATE_PATTERNS  = re.compile(r'CRE|REG|INS|FIRST|OPEN|START|BEGIN|OCCUR|INIT|ENTR', re.IGNORECASE)
VARCHAR_DATE_PATTERNS = [
    (8,  r'^\d{8}$',                                 '%Y%m%d'),
    (12, r'^\d{12}$',                                '%Y%m%d%H%M'),
    (14, r'^\d{14}$',                                '%Y%m%d%H%M%S'),
    (16, r'^\d{16}$',                                '%Y%m%d%H%M%S%f'),
    (17, r'^\d{17,}$',                               '%Y%m%d%H%M%S%f'),
    (10, r'^\d{4}-\d{2}-\d{2}$',                    '%Y-%m-%d'),
    (19, r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$', '%Y-%m-%d %H:%M:%S'),
]

def sanitize_val(v):
    """Excel 에 쓸 수 없는 제어문자 제거 (IllegalCharacterError 방지)"""
    if isinstance(v, str):
        return ILLEGAL_CHARS_RE.sub('', v)
    return v

def sanitize_df(df):
    try:
        return df.map(sanitize_val)        # pandas >= 2.1
    except AttributeError:
        return df.applymap(sanitize_val)   # pandas < 2.1

def _decode_val(v):
    """Oracle bytes → str (CP949 fallback)"""
    if isinstance(v, bytes):
        try:
            return v.decode('utf-8')
        except UnicodeDecodeError:
            return v.decode('cp949', errors='replace')
    return v

def fmt_gb(b):
    if not b: return '0.0GB'
    gb = b / (1024 ** 3)
    if gb <= 0:  return '0.0GB'
    if gb < 0.1: return '0.1GB'
    return f'{gb:.1f}GB'

def parse_cell(ref):
    m = re.match(r'^([A-Za-z]+)(\d+)$', ref.strip())
    if not m: raise ValueError(f'잘못된 셀: {ref}')
    return column_index_from_string(m.group(1)), int(m.group(2))

def fetchall_dict(cur):
    cols = [d[0].lower() for d in cur.description]
    return [dict(zip(cols, (_decode_val(v) for v in row))) for row in cur.fetchall()]

def _read_lob(val):
    if val is None: return None
    if hasattr(val, 'read'):
        try:
            data = val.read()
            return _decode_val(data) if isinstance(data, bytes) else data
        except Exception:
            return str(val)
    return _decode_val(val)

print('utilities OK')

In [ ]:
# 셀 4: DB 쿼리 함수
USAGE_SQL = '''
WITH
q3_select AS (
    SELECT  sp.object_name           AS table_name,
            MAX(sa.last_active_time) AS last_select_time
    FROM    v$sql_plan sp
    JOIN    v$sqlarea  sa ON sa.sql_id = sp.sql_id
    WHERE   sp.object_owner = USER
      AND   sp.object_name  IS NOT NULL
      AND   sp.object_type  LIKE 'TABLE%%'
    GROUP   BY sp.object_name
),
ash_access AS (
    SELECT  o.object_name                                              AS table_name,
            COUNT(CASE WHEN ash.sql_opname = 'SELECT' THEN 1 END)     AS ash_select,
            COUNT(CASE WHEN ash.sql_opname IN (
                       'INSERT','UPDATE','DELETE','MERGE') THEN 1 END) AS ash_dml
    FROM    dba_hist_active_sess_history ash
    JOIN    dba_objects o
            ON  o.object_id  = ash.current_obj#
            AND o.owner       = USER
            AND o.object_type = 'TABLE'
    WHERE   ash.sample_time >= SYSDATE - 8
      AND   ash.current_obj# > 0
    GROUP   BY o.object_name
),
dml_hist AS (
    SELECT  table_name,
            SUM(inserts) AS inserts,
            SUM(updates) AS updates,
            SUM(deletes) AS deletes
    FROM    user_tab_modifications
    WHERE   partition_name IS NULL
    GROUP   BY table_name
),
master_chk AS (
    SELECT  table_name,
            CASE WHEN REGEXP_LIKE(table_name,
                 'CODE|COCD|_CD$|_CD_|MST|MASTER|BASE|STD|CMM|COMMON|' ||
                 'TYPE|KIND|STAT|STATUS|GRP|GROUP|CLASS|CATG|CONFIG|PARAM|META|' ||
                 'MENU|ROLE|AUTH|PERMISSION|POLICY', 'i')
                 THEN 'Y' ELSE 'N' END AS is_master
    FROM    user_tables
)
SELECT
    t.table_name,
    NVL(t.num_rows, -1) AS num_rows,
    CASE
        WHEN ash.ash_select > 0                                      THEN 'Y (ASH-SELECT확인)'
        WHEN ash.ash_dml    > 0                                      THEN 'Y (ASH-DML확인)'
        WHEN q3.table_name IS NOT NULL                               THEN 'Y (Q3-SELECT흔적)'
        WHEN NVL(m.inserts,0)+NVL(m.updates,0)+NVL(m.deletes,0) > 0 THEN 'Y (DML이력)'
        WHEN t.num_rows = 0                                          THEN 'N'
        WHEN t.num_rows IS NULL                                      THEN '확인필요 (통계미수집)'
        WHEN t.num_rows > 0 AND mc.is_master = 'Y'                  THEN '확인필요 (마스터추정)'
        WHEN t.num_rows > 0                                          THEN '확인필요'
        ELSE 'N'
    END AS usage_flag
FROM    user_tables t
LEFT JOIN dml_hist   m   ON  m.table_name  = t.table_name
LEFT JOIN q3_select  q3  ON  q3.table_name = t.table_name
LEFT JOIN ash_access ash ON ash.table_name = t.table_name
LEFT JOIN master_chk mc  ON  mc.table_name = t.table_name
'''

def load_usage_map(conn):
    cur = conn.cursor()
    try:
        try:
            cur.execute('BEGIN DBMS_STATS.FLUSH_DATABASE_MONITORING_INFO; END;')
        except Exception:
            pass
        cur.execute(USAGE_SQL)
        rows = fetchall_dict(cur)
        return {r['table_name']: r['usage_flag'] for r in rows}
    except Exception as e:
        print(f'  [WARN] 사용여부 쿼리 실패: {e}')
        return {}
    finally:
        cur.close()

def get_table_list(conn):
    cur = conn.cursor()
    cur.execute('''
        SELECT t.table_name,
               NVL(tc.comments, '-') AS comments,
               t.num_rows,
               o.created             AS created_dt
        FROM   user_tables      t
        LEFT JOIN user_tab_comments tc ON tc.table_name = t.table_name
        LEFT JOIN user_objects      o  ON  o.object_name = t.table_name
                                       AND o.object_type  = 'TABLE'
        ORDER BY t.table_name
    ''')
    return fetchall_dict(cur)

def get_column_count(conn, table_name):
    cur = conn.cursor()
    cur.execute('SELECT COUNT(*) FROM user_tab_columns WHERE table_name = :1', [table_name])
    row = cur.fetchone()
    cur.close()
    return row[0] if row else 0

def get_size_bytes(conn, table_name):
    cur = conn.cursor()
    cur.execute('''
        SELECT NVL(SUM(bytes), 0)
        FROM (
            SELECT bytes FROM user_segments WHERE segment_name = :1
            UNION ALL
            SELECT s.bytes
            FROM   user_lobs l
            JOIN   user_segments s ON s.segment_name = l.segment_name
            WHERE  l.table_name = :2
        )
    ''', [table_name, table_name])
    row = cur.fetchone()
    cur.close()
    return row[0] if row else 0

def get_monthly_avg_insert(conn, table_name):
    cur = conn.cursor()
    cur.execute('''
        SELECT NVL(SUM(inserts), 0), MIN(timestamp), MAX(timestamp)
        FROM   user_tab_modifications
        WHERE  table_name = :1 AND partition_name IS NULL
    ''', [table_name])
    row = cur.fetchone()
    cur.close()
    if not row or row[0] == 0: return '0'
    total_ins, min_dt, max_dt = row
    months = max(1, round((max_dt - min_dt).days / 30)) if (min_dt and max_dt and min_dt != max_dt) else 1
    avg = round(total_ins / months)
    return f'{avg:,}' if avg >= 10000 else str(avg)

def _try_parse_varchar_date(val):
    s = str(val).strip()
    for min_len, pattern, _ in VARCHAR_DATE_PATTERNS:
        if len(s) >= min_len and re.match(pattern, s):
            return True
    try:
        dateutil_parser.parse(s, yearfirst=True, ignoretz=True)
        return True
    except Exception:
        return False

def _is_varchar_date_col(conn, table_name, col_name):
    cur = conn.cursor()
    try:
        cur.execute(f'SELECT {col_name} FROM {table_name} WHERE ROWNUM <= 100')
        rows = cur.fetchall()
        if not rows: return False
        vals = [r[0] for r in rows if r[0] is not None]
        if not vals: return False
        return sum(1 for v in vals if _try_parse_varchar_date(str(v))) / len(vals) >= 0.8
    except Exception:
        return False
    finally:
        cur.close()

def detect_date_column(conn, table_name):
    cur = conn.cursor()
    cur.execute('''
        SELECT column_name, data_type
        FROM   user_tab_columns
        WHERE  table_name = :1
        ORDER  BY column_id
    ''', [table_name])
    cols = cur.fetchall()
    cur.close()
    date_pat_cols, date_cols, varchar_cols = [], [], []
    for col_name, data_type in cols:
        if data_type == 'DATE' or data_type.startswith('TIMESTAMP'):
            (date_pat_cols if CREATE_PATTERNS.search(col_name) else date_cols).append(col_name)
        elif 'CHAR' in data_type:
            varchar_cols.append(col_name)
    if date_pat_cols: return date_pat_cols[0]
    if date_cols:     return date_cols[0]
    for col_name in varchar_cols:
        if _is_varchar_date_col(conn, table_name, col_name):
            return col_name
    return None

def get_first_date(conn, table_name, num_rows, date_col, created_dt):
    if num_rows == 0: return '-'
    if date_col:
        cur = conn.cursor()
        try:
            cur.execute(f'SELECT MIN({date_col}) FROM {table_name}')
            row = cur.fetchone()
            if row and row[0] is not None:
                val = row[0]
                if isinstance(val, datetime):
                    return val.strftime('%Y-%m-%d')
                s = str(val).strip()
                if re.match(r'^\d{8}', s):
                    try: return datetime.strptime(s[:8], '%Y%m%d').strftime('%Y-%m-%d')
                    except Exception: pass
                try: return dateutil_parser.parse(s, ignoretz=True).strftime('%Y-%m-%d')
                except Exception: return s[:10]
        except Exception:
            pass
        finally:
            cur.close()
    if created_dt:
        return created_dt.strftime('%Y-%m-%d') if isinstance(created_dt, datetime) else str(created_dt)[:10]
    return '-'

def fetch_sample(conn, table_name, date_col):
    if date_col:
        sql = (f'SELECT * FROM (SELECT * FROM {table_name} ORDER BY {date_col} DESC)'
               f' WHERE ROWNUM <= {SAMPLE_ROWS}')
    else:
        sql = f'SELECT * FROM {table_name} WHERE ROWNUM <= {SAMPLE_ROWS}'
    cur = conn.cursor()
    try:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = [tuple(_read_lob(v) for v in row) for row in cur.fetchall()]
        return pd.DataFrame(rows, columns=cols)
    except Exception as e:
        return pd.DataFrame({'ERROR': [str(e)]})
    finally:
        cur.close()

print('DB functions OK')

In [ ]:
# 셀 5: Oracle 클라이언트 초기화 + DB 접속
oracledb.init_oracle_client(lib_dir=ORACLE_CLIENT_LIB)
conn = oracledb.connect(
    user=DB_USER, password=DB_PASSWORD,
    dsn=f'{DB_HOST}:{DB_PORT}/{DB_SERVICE}'
)
print('접속 성공')

In [ ]:
# 셀 6: 사용여부 맵 일괄 조회
print('사용여부 쿼리 실행 중...')
usage_map = load_usage_map(conn)
print(f'완료: {len(usage_map)}개 테이블 사용여부 로드')

In [ ]:
# 셀 7: 테이블 데이터 수집 (오래 걸림)
tables = get_table_list(conn)
total  = len(tables)
print(f'총 {total}개 테이블 수집 시작...')

records = []
for idx, tbl in enumerate(tables, start=1):
    tname      = tbl['table_name']
    comment    = tbl['comments'] or '-'
    num_rows   = tbl['num_rows']
    created_dt = tbl['created_dt']
    nr         = int(num_rows) if num_rows is not None else None

    usage_flag = usage_map.get(tname, '확인필요')
    date_col   = detect_date_column(conn, tname)
    first_date = get_first_date(conn, tname, nr if nr is not None else -1, date_col, created_dt)
    monthly    = get_monthly_avg_insert(conn, tname)
    col_count  = get_column_count(conn, tname)
    size_bytes = get_size_bytes(conn, tname)

    records.append({
        '번호':               idx,
        '시스템명':           SYSTEM_NAME,
        '테이블명':           tname,
        '테이블설명':         comment,
        '사용여부':           usage_flag,
        '데이터보관최초일자': first_date,
        '1개월평균데이터생성건수': monthly,
        '칼럼수(개)':         col_count,
        '전체테이블용량':     fmt_gb(size_bytes),
        '_date_col':          date_col,
    })

    if idx % 50 == 0 or idx == total:
        print(f'  {idx}/{total}  최근: {tname}  최초일={first_date}')

print(f'수집 완료: {len(records)}개')

In [ ]:
# 셀 8: 체크포인트 저장 (셀 7 완료 직후 실행)
with open(CHECKPOINT_FILE, 'wb') as f:
    pickle.dump(records, f)
print(f'체크포인트 저장: {CHECKPOINT_FILE}  ({len(records)}개 레코드)')

In [ ]:
# 셀 9: 체크포인트 로드 (재실행 시 셀 7 건너뛰고 이 셀 실행)
with open(CHECKPOINT_FILE, 'rb') as f:
    records = pickle.load(f)
print(f'체크포인트 로드 완료: {len(records)}개 레코드')

In [ ]:
# 셀 10: 메인 데이터를 Excel 템플릿에 기록
if not os.path.isfile(TEMPLATE_EXCEL):
    raise FileNotFoundError(f'파일 없음: {TEMPLATE_EXCEL}')

ts     = datetime.now().strftime('%H%M%S')
backup = TEMPLATE_EXCEL.replace('.xlsx', f'_backup_{ts}.xlsx')
shutil.copy2(TEMPLATE_EXCEL, backup)
print(f'백업: {backup}')

start_col, start_row = parse_cell(START_CELL)
LINK_FONT = Font(color='0563C1', underline='single', bold=True)
COLS = [
    '번호', '시스템명', '테이블명', '테이블설명', '사용여부',
    '데이터보관최초일자', '1개월평균데이터생성건수', '칼럼수(개)', '전체테이블용량',
]

wb = load_workbook(TEMPLATE_EXCEL)
ws = wb.active

for row_offset, rec in enumerate(records):
    excel_row = start_row + row_offset
    for col_offset, key in enumerate(COLS):
        ws.cell(row=excel_row, column=start_col + col_offset).value = sanitize_val(rec[key])

prefix = "#'"
suffix = "'!A1"
for row_offset, rec in enumerate(records):
    excel_row      = start_row + row_offset
    cell           = ws.cell(row=excel_row, column=start_col)
    cell.value     = rec['번호']
    cell.hyperlink = prefix + str(rec['번호']) + suffix
    cell.font      = LINK_FONT

wb.save(TEMPLATE_EXCEL)
print(f'메인 데이터 저장 완료: {TEMPLATE_EXCEL}')

In [ ]:
# 셀 11: 샘플 시트 생성 (pandas DataFrame.to_excel 사용 — 오래 걸림)
# ※ DB 접속이 끊겼다면 셀 5를 먼저 재실행하세요
print(f'샘플 시트 {len(records)}개 생성 중...')

with pd.ExcelWriter(TEMPLATE_EXCEL, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for rec in records:
        tname    = rec['테이블명']
        date_col = rec['_date_col']
        df       = fetch_sample(conn, tname, date_col)
        df       = sanitize_df(df)
        df.to_excel(writer, sheet_name=str(rec['번호']), index=False)
        if rec['번호'] % 100 == 0:
            print(f'  {rec["번호"]}/{len(records)}: {tname}')

print(f'완료: {TEMPLATE_EXCEL}')